# 01 业务问题与数据理解

## 本步目标

1. 看懂“申请—审批—用信”链路。
2. 搞清楚 `applications` 和 `repayments` 两张表的粒度。
3. 明确本次专项要回答的问题和 3 个初始假设。

## 业务问题

2026-09 审批通过率从 29.4% 降到 22.9%，下降约 6.5 个百分点。我们要判断：

- 是渠道结构变化，还是同一渠道审批变严？
- 哪些渠道贡献最大？
- 通过率下降是否对应更好的风险表现？

## 你要记住的口径

- 审批通过率 =（Approved + Unused offer）/ 全部进件。
- 用信率 = Approved /（Approved + Unused offer）。
- `applications` 一行一笔历史申请；`repayments` 一行一期还款。
- 金额单位是“万元（归一化）”。
- 日期是以 2026-09-30 为锚点映射的业务日历。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from IPython.display import display
from src.db import query_df
from src.charts import monthly_trend, channel_bar, contribution_bar, risk_scatter

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 50)
FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

## 1. 两张表各有多少行？

In [2]:
display(query_df("SELECT COUNT(*) AS 申请行数 FROM `v_申请明细`"))
display(query_df("SELECT COUNT(*) AS 还款行数 FROM `v_还款明细`"))

,申请行数
0,1670214


,还款行数
0,13605401


## 2. 数据覆盖什么时间范围？

In [3]:
display(query_df('''
SELECT
    MIN(decision_date) AS 最早申请日,
    MAX(decision_date) AS 最晚申请日,
    COUNT(DISTINCT DATE_FORMAT(decision_date, '%Y-%m')) AS 覆盖月份数
FROM `v_申请明细`
'''))

,最早申请日,最晚申请日,覆盖月份数
0,2018-09-30,2026-09-29,97


## 3. 审批状态长什么样？

In [4]:
display(query_df('''
SELECT status AS 审批状态, COUNT(*) AS 笔数,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS 占比
FROM `v_申请明细`
GROUP BY status
ORDER BY 笔数 DESC
'''))

,审批状态,笔数,占比
0,Approved,1036781,0.6207
1,Canceled,316319,0.1894
2,Refused,290678,0.1740
3,Unused offer,26436,0.0158


## 4. 渠道长什么样？先不要急着下结论，只看规模。

In [5]:
display(query_df('''
SELECT COALESCE(channel_type, '未知渠道') AS 渠道类型,
       COUNT(*) AS 进件量,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS 进件占比,
       ROUND(SUM(status IN ('Approved','Unused offer')) / COUNT(*), 4) AS 审批通过率
FROM `v_申请明细`
GROUP BY 1
ORDER BY 进件量 DESC
'''))

,渠道类型,进件量,进件占比,审批通过率
0,Credit and cash offices,719968,0.4311,0.4022
1,Country-wide,494690,0.2962,0.8635
2,Stone,212083,0.1270,0.8960
3,Regional / Local,108528,0.0650,0.8948
4,Contact center,71297,0.0427,0.3537
5,AP+ (Cash loan),57046,0.0342,0.5476
6,Channel of corporate sales,6150,0.0037,0.4307
7,Car dealer,452,0.0003,0.6327


## 本步结论

数据覆盖 2018-09 至 2026-09，共 1,670,214 笔历史申请；2026-09 进件 36,012 笔，审批状态以 Canceled、Approved、Refused 为主。后续分析统一使用“全部进件”作为通过率分母。